# kernel

> Isolated agent runtime: a persistent [clikernel](https://github.com/AnswerDotAI/clikernel)
> worker that paar supervises as a **separate process** from the owner's kernel. Agent code runs
> here, never in the owner's interpreter, so it cannot touch owner objects by construction. The
> agent reads owner state read-only through paar's snapshot API and does its compute in this worker.
> paar's restriction policy is applied *before* code is forwarded, so guardrails hold even though
> the worker itself is a plain clikernel.

In [ ]:
#| default_exp kernel

In [ ]:
#| hide
from nbdev.showdoc import *

## AgentKernel

One persistent worker per `AgentKernel`, launched with paar's own interpreter
(`sys.executable -m clikernel.cli`) so it inherits the same environment (paar, and whatever else
is installed). It speaks clikernel's stdin/stdout stream protocol: a bare `--` opens a multiline
cell, closed by the session delimiter; each response is an ack line `.`, the rendered output, then
the delimiter again.

In [ ]:
#| export
import subprocess, os, re, sys, threading, atexit
from paar.agent import policy_inspectors, run_inspectors, RuleBlock
from paar.runtime import ExecResult
try: from clikernel.base import _MARKER
except Exception: _MARKER = "loading complete. session delimiter:"

class AgentKernel:
    "Supervises one persistent clikernel worker as the agent's isolated runtime -- a separate process from the owner's kernel, so agent code cannot touch owner objects. Read owner state via paar's read-only snapshot API; compute here."
    def __init__(self,
                 name='agent',   # env label for the worker's own paar server / registry entry
                 argv=None):     # worker command (default: this interpreter's clikernel worker)
        self.name = name
        self.argv = argv or [sys.executable, '-m', 'clikernel.cli']
        self.proc = self.delim = None
        self.lock = threading.Lock()

    def alive(self):
        "Is the worker process running?"
        return self.proc is not None and self.proc.poll() is None

    def _readline(self):
        return self.proc.stdout.readline().decode('utf-8', 'replace').rstrip('\n')

    def start(self):
        "Launch the worker and read its banner up to the session delimiter."
        if self.alive(): return self
        env = {**os.environ, 'PYTHONUNBUFFERED': '1', 'PAAR_ENV_NAME': self.name, 'PAAR_AGENT_WORKER': '1'}
        self.proc = subprocess.Popen(self.argv, stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                                     stderr=subprocess.STDOUT, env=env)
        while True:
            line = self._readline()
            if line == '' and not self.alive(): raise RuntimeError('clikernel worker failed to start')
            if line == _MARKER: break
        self.delim = self._readline()
        atexit.register(self.stop)
        return self

    def _run_raw(self, code):
        self.proc.stdin.write(f'--\n{code}\n{self.delim}\n'.encode()); self.proc.stdin.flush()
        lines = []
        while True:
            line = self._readline()
            if line == '' and not self.alive(): return '<kernel process died>'
            if line == self.delim: break
            lines.append(line)
        if lines and lines[0] == '.': lines = lines[1:]   # drop the ack echo
        return '\n'.join(lines)

    def run(self, code):
        "Run `code` in the worker (starting it if needed); return the rendered output text."
        with self.lock:
            if not self.alive(): self.start()
            return self._run_raw(code)

    def serve_paar(self, port=8001):
        "Have the worker ride a paar inspector on its own namespace so the owner can watch it live."
        return self.run(f'import paar.fasthtml as _pf; _pf.serve(port={port}, name={self.name!r})')

    def stop(self):
        "Terminate the worker (graceful `exit`, then kill)."
        if self.alive():
            try: self.proc.stdin.write(b'exit\n'); self.proc.stdin.flush()
            except Exception: pass
            try: self.proc.wait(timeout=3)
            except Exception: self.proc.kill()

def guarded_run(kernel, code, owner_names=None):
    "Apply paar's restriction policy, then run in the isolated worker. Returns an ExecResult; a blocked cell never reaches the worker."
    owner_names = owner_names or (lambda: set())
    try: note = run_inspectors(policy_inspectors(owner_names), code)
    except RuleBlock as e: return ExecResult(ok=False, error=f'blocked: {e}')
    return ExecResult(ok=True, stdout=note + kernel.run(code))

## paar-exec CLI

Agents drive the isolated kernel by POSTing to a live paar server's `/kernel/api/exec` (the transport,
since MCP registration is disabled here). `paar-mcp` and the `/agent/api` overlay are left untouched.

In [ ]:
#| export
def _resolve_base(url=None, env=None):
    "Base URL of a live paar server: explicit url > env name in the registry > first live one."
    import paar.registry as R
    if url: return url.rstrip('/')
    envs = R.active()
    if env:
        for e in envs:
            if e['name'] == env: return e['base']
        raise SystemExit(f"no live paar env named {env!r}; live: {[e['name'] for e in envs]}")
    if not envs: raise SystemExit('no live paar server; start one with paar.fasthtml.serve() or inspector()')
    return envs[0]['base']

def main(argv=None):
    "Console entry `paar-exec [--env NAME | --url URL] [CODE]`: run CODE (or stdin) in a live paar server's process-isolated agent kernel."
    import argparse, sys, httpx
    p = argparse.ArgumentParser(prog='paar-exec', description="run code in a live paar server's process-isolated agent kernel (/kernel/api/exec)")
    p.add_argument('code', nargs='?', help='code to run (reads stdin if omitted)')
    p.add_argument('--env', default=None, help='pick a live paar env by name')
    p.add_argument('--url', default=None, help='paar server base URL (default: discover via the registry)')
    a = p.parse_args(argv)
    code = a.code if a.code is not None else sys.stdin.read()
    r = httpx.post(_resolve_base(a.url, a.env) + '/kernel/api/exec', data={'code': code}, timeout=120.0)
    try: d = r.json()
    except Exception: raise SystemExit(f'HTTP {r.status_code}: {r.text[:300]}')
    if d.get('stdout'): print(d['stdout'])
    if d.get('error'): print(d['error'], file=sys.stderr); raise SystemExit(1)

## Tests

In [ ]:
k = AgentKernel().start()
try:
    assert k.alive()
    assert '2' in k.run('1 + 1')                      # basic exec
    assert k.run('nums = list(range(5))') is not None
    assert '10' in k.run('sum(nums)')                 # state persists across calls
    # guardrails applied before forwarding: escapes never reach the worker
    r = guarded_run(k, 'import os; os.system("echo LEAK")')
    assert (not r.ok) and r.error.startswith('blocked:'), r
    # a benign cell passes and runs
    r2 = guarded_run(k, 'y = 40 + 2\nprint(y)')
    assert r2.ok and '42' in r2.stdout, r2
finally:
    k.stop()
assert not k.alive()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()